# MP3 AI Türkçe Dublaj — Colab
**Runtime > Change runtime type > GPU** seçin.

## 0. Colab Secrets kurulumu (BİR KEZ)
Sol kenardaki **🔑 (anahtar)** simgesine tıkla → **Add new secret**. İki secret ekle ve her birinde **Notebook access** anahtarını AÇ:
- `GITHUB_TOKEN` → private repoyu çekmek/push için (GitHub fine-grained PAT, 0xR0/dublaj deposuna Contents: Read and write)
- `HF_TOKEN` → (opsiyonel) pyannote için ücretsiz HF token. Eklemezsen speechbrain kullanılır.

Bir kez girince her oturumda otomatik okunur; tekrar yazmana gerek kalmaz.

## 1. Repo (private) + sistem bağımlılıkları

In [ ]:
from google.colab import userdata
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
!apt-get -qq install -y ffmpeg
!pip install -q yt-dlp
%cd /content
!rm -rf dublaj
!git clone https://{GITHUB_TOKEN}@github.com/0xR0/dublaj.git
%cd dublaj

## 2. Python bağımlılıkları (birkaç dakika)

In [ ]:
!pip install -q -r requirements.txt

## 3. Ortam değişkenleri (Secrets'tan otomatik)
HF_TOKEN secret'ı varsa pyannote, yoksa speechbrain kullanılır.

In [ ]:
import os
from google.colab import userdata
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN') or ''
except Exception:
    os.environ['HF_TOKEN'] = ''
os.environ['COQUI_TOS_AGREED'] = '1'
print('HF_TOKEN ayarli mi:', bool(os.environ['HF_TOKEN']))

## 4-A. Girdi: YouTube linkinden ses indir

In [ ]:
YT_URL = "https://youtu.be/2hsSEWguOQE"  # <-- linki degistir
!yt-dlp -x --audio-format mp3 --audio-quality 0 -o "input/yt_audio.%(ext)s" "$YT_URL"
INPUT = "input/yt_audio.mp3"
print("Girdi:", INPUT)

## 4-B. (Alternatif) Kendi dosyanı yükle

In [ ]:
from google.colab import files
import shutil
up = files.upload()
name = list(up.keys())[0]
shutil.move(name, f'input/{name}')
INPUT = f'input/{name}'
print('Girdi:', INPUT)

## 5. Dublajı çalıştır
İlk sefer modeller iner (XTTS ~1.8GB). Bayraklar: `--no-background`, `--tts edge`, `--diarizer speechbrain|pyannote`.

In [ ]:
!python dub.py "$INPUT"

## 6. Sonucu incele

In [ ]:
!ls -la output/
print('--- run.log (son 40 satir) ---')
!tail -n 40 logs/run.log

## 7. Sonucu indir

In [ ]:
from google.colab import files
files.download('output/output_dubbed.mp3')

## 8. (Opsiyonel) Sonucu GitHub'a geri gönder
Termux tarafında commitler + log + çıktı incelenebilsin diye.

In [ ]:
from google.colab import userdata
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
!git config user.email "colab@dublaj"
!git config user.name "colab"
!git add -f output/output_dubbed.mp3 logs/run.log
!git commit -m "colab: dublaj sonucu + loglar" || echo "degisiklik yok"
!git push https://{GITHUB_TOKEN}@github.com/0xR0/dublaj.git HEAD:colab-results